# NFL Public Data Warehouse — Collection & Exploration

**Goal:** Collect publicly accessible NFL data into separate pandas DataFrames, inventory source coverage, and export results for analysis.

This notebook is deliberately resilient: individual sources can fail without stopping the whole run. Public availability does **not** automatically mean unrestricted reuse. Review each source's terms, licenses, rate limits, and attribution requirements before publishing or redistributing data.

## Scope and limits
- Core bulk collection uses **nflreadpy / nflverse**, which offers maintained downloadable datasets.
- An inventory cell queries GitHub's public API for the complete current nflverse release/asset catalog.
- An optional ESPN scoreboard example demonstrates a public endpoint, but unofficial endpoints can change without notice.
- Official NFL pages, Pro Football Reference, Kaggle, GitHub projects, college-football sources, and licensed providers are listed in the source registry for follow-up.
- Raw RFID/player tracking at 10 Hz, complete All-22 video, private team data, some charting data, and some authenticated NFL endpoints are **not** all publicly downloadable. Public Next Gen Stats summaries are not the same as raw tracking data.

The notebook loads broad historical coverage by default where supported. For large datasets, run one category at a time or change `SEASONS`.

## 0. Source registry

A practical inventory of the main public or discoverable source families. Access method and coverage can change; verify before relying on a source.

In [ ]:
import pandas as pd

source_registry = pd.DataFrame([
    {"source": "nflverse data releases", "url": "https://github.com/nflverse/nflverse-data/releases", "data": "PBP, schedules, stats, rosters, players, snaps, draft, combine, NGS, contracts, depth charts, other releases", "access": "Direct downloadable files / nflreadpy", "key_or_login": "Usually no", "notes": "Core bulk source; dataset coverage differs by season."},
    {"source": "nflreadpy documentation", "url": "https://nflreadpy.nflverse.com/", "data": "Python loaders for nflverse datasets", "access": "Python package", "key_or_login": "No", "notes": "Polars output; convert to pandas."},
    {"source": "NFL official statistics", "url": "https://www.nfl.com/stats/", "data": "Official player/team stats and leaderboards", "access": "Website; APIs may be authenticated/restricted", "key_or_login": "Depends", "notes": "Do not assume internal website endpoints are an open API."},
    {"source": "NFL Next Gen Stats", "url": "https://nextgenstats.nfl.com/", "data": "Published tracking-derived metrics", "access": "Website; selected summaries mirrored by nflverse", "key_or_login": "Some endpoints require auth", "notes": "Public summaries are not raw tracking feeds."},
    {"source": "NFL Football Operations / Big Data Bowl", "url": "https://operations.nfl.com/programs-initiatives/innovation/big-data-bowl", "data": "Special competition datasets, tracking samples, research", "access": "Competition-specific download / Kaggle", "key_or_login": "May require account or eligibility", "notes": "Historical competitions can expose data unavailable elsewhere."},
    {"source": "Pro Football Reference", "url": "https://www.pro-football-reference.com/", "data": "Historical game logs, box scores, player/team stats, draft, combine", "access": "Website; some data mirrored in open projects", "key_or_login": "Some products paid", "notes": "Respect robots.txt, terms, and rate limits; avoid aggressive scraping."},
    {"source": "ESPN NFL", "url": "https://www.espn.com/nfl/", "data": "Scores, schedules, box scores, play-by-play, QBR, injuries, transactions", "access": "Website and some undocumented public JSON endpoints", "key_or_login": "Often no key, but undocumented", "notes": "Endpoint stability and reuse rights are not guaranteed."},
    {"source": "Kaggle NFL datasets", "url": "https://www.kaggle.com/datasets?search=nfl", "data": "Community datasets, Big Data Bowl data, historical/statistical collections", "access": "Manual download or Kaggle API", "key_or_login": "Account/API token often needed", "notes": "Check dataset owner, license, freshness, and provenance."},
    {"source": "GitHub public repositories", "url": "https://github.com/search?q=nfl+data&type=repositories", "data": "Community datasets, scrapers, schemas, historical archives", "access": "Git / GitHub API / release files", "key_or_login": "Usually no; API rate limits apply", "notes": "Verify provenance and licenses individually."},
    {"source": "CollegeFootballData", "url": "https://collegefootballdata.com/", "data": "College stats, schedules, recruiting, teams and draft-prospect context", "access": "Documented API", "key_or_login": "API key", "notes": "Useful for NFL prospect/college context; not NFL game data."},
    {"source": "SportsDataIO", "url": "https://sportsdata.io/", "data": "Schedules, scores, rosters, injuries, odds and stats", "access": "Commercial API", "key_or_login": "Usually paid/keyed", "notes": "Not a free bulk source."},
    {"source": "Sportradar", "url": "https://sportradar.com/", "data": "Scores, play-by-play, statistics and feeds", "access": "Commercial API", "key_or_login": "Paid/keyed", "notes": "Not public bulk data."},
    {"source": "FantasyPros", "url": "https://www.fantasypros.com/nfl/", "data": "Rankings, projections, fantasy stats", "access": "Website / licensed feeds", "key_or_login": "Depends", "notes": "Use only permitted download/API methods."},
    {"source": "OverTheCap", "url": "https://overthecap.com/", "data": "Contracts, cap hits, salary-cap and roster transactions", "access": "Website; selected data mirrored by nflverse", "key_or_login": "Depends", "notes": "Contract data may be estimates and is not official league payroll."},
    {"source": "NFL GSIS / official APIs", "url": "https://www.nfl.com/", "data": "Official data services and internal league feeds", "access": "Some services require authentication/authorization", "key_or_login": "Often yes", "notes": "Do not bypass authentication or access controls."},
])
display(source_registry)

## 1. Install dependencies

Run this cell once in your notebook environment. Restart the kernel if prompted.

In [ ]:
%pip install -q nflreadpy pandas pyarrow requests tqdm

## 2. Configuration

In [ ]:
from pathlib import Path
import json, time, warnings, requests
from datetime import datetime, timezone

OUTPUT_DIR = Path("nfl_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# True means all seasons available for that dataset; may download large files.
ALL_SEASONS = True
# If ALL_SEASONS=False, set a list such as [2022, 2023, 2024, 2025, 2026].
SEASONS = [2022, 2023, 2024, 2025, 2026]
SAVE_PARQUET = True
SAVE_CSV = False
REQUEST_TIMEOUT = 45

run_log = []
def log_result(source, status, detail=""):
    run_log.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source": source, "status": status, "detail": str(detail)[:1000]
    })

## 3. Load maintained nflverse datasets into separate pandas DataFrames

Each dataset is attempted independently. Depending on package version, a few optional loaders may be absent or unavailable. Failures are recorded in `collection_status` rather than stopping the notebook.

In [ ]:
import nflreadpy as nfl

def as_pandas(obj):
    """Convert Polars/pandas-like objects to pandas without assuming a single return type."""
    if obj is None:
        return pd.DataFrame()
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, "to_pandas"):
        return obj.to_pandas()
    try:
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame()

def chosen_seasons():
    return True if ALL_SEASONS else SEASONS

# Loader name -> callable arguments.
# Optional datasets can be removed from this mapping to reduce download volume.
loader_specs = {
    "pbp": ("load_pbp", {"seasons": chosen_seasons()}),
    "schedules": ("load_schedules", {"seasons": chosen_seasons()}),
    "player_stats": ("load_player_stats", {"seasons": chosen_seasons()}),
    "team_stats": ("load_team_stats", {"seasons": chosen_seasons()}),
    "players": ("load_players", {}),
    "rosters": ("load_rosters", {"seasons": chosen_seasons()}),
    "weekly_rosters": ("load_rosters_weekly", {"seasons": chosen_seasons()}),
    "teams": ("load_teams", {}),
    "participation": ("load_participation", {"seasons": chosen_seasons()}),
    "draft_picks": ("load_draft_picks", {"seasons": chosen_seasons()}),
    "combine": ("load_combine", {"seasons": chosen_seasons()}),
    "nextgen_passing": ("load_nextgen_stats", {"seasons": chosen_seasons(), "stat_type": "passing"}),
    "nextgen_receiving": ("load_nextgen_stats", {"seasons": chosen_seasons(), "stat_type": "receiving"}),
    "nextgen_rushing": ("load_nextgen_stats", {"seasons": chosen_seasons(), "stat_type": "rushing"}),
    "depth_charts": ("load_depth_charts", {"seasons": chosen_seasons()}),
    "injuries": ("load_injuries", {"seasons": chosen_seasons()}),
    "snap_counts": ("load_snap_counts", {"seasons": chosen_seasons()}),
    "contracts": ("load_contracts", {}),
    "pfr_advanced_stats": ("load_pfr_advstats", {"seasons": chosen_seasons()}),
    "espn_qbr": ("load_espn_qbr", {"seasons": chosen_seasons()}),
    "officials": ("load_officials", {}),
    "trades": ("load_trades", {}),
    "fantasy_player_ids": ("load_ff_playerids", {}),
    "fantasy_rankings": ("load_ff_rankings", {}),
    "fantasy_opportunity": ("load_ff_opportunity", {}),
    "ftn_charting": ("load_ftn_charting", {"seasons": chosen_seasons()}),
}

dataframes = {}
collection_status = []

for df_name, (function_name, kwargs) in loader_specs.items():
    loader = getattr(nfl, function_name, None)
    if loader is None:
        msg = f"Loader {function_name} is not present in this installed nflreadpy version."
        collection_status.append({"dataset": df_name, "status": "unavailable", "rows": None, "columns": None, "detail": msg})
        log_result(df_name, "unavailable", msg)
        continue
    try:
        result = loader(**kwargs)
        df = as_pandas(result)
        dataframes[df_name] = df
        collection_status.append({"dataset": df_name, "status": "success", "rows": len(df), "columns": len(df.columns), "detail": ""})
        log_result(df_name, "success", f"{len(df)} rows, {len(df.columns)} columns")
        print(f"{df_name:24s} {len(df):>10,} rows x {len(df.columns):>4} columns")
    except Exception as e:
        collection_status.append({"dataset": df_name, "status": "failed", "rows": None, "columns": None, "detail": repr(e)})
        log_result(df_name, "failed", repr(e))
        print(f"{df_name:24s} FAILED: {e}")

collection_status = pd.DataFrame(collection_status)
display(collection_status)

## 4. Create easy-to-use variables for each DataFrame

Use `pbp`, `schedules`, `players`, etc. directly in later analysis. `dataframes` remains the authoritative dictionary.

In [ ]:
for _name, _df in dataframes.items():
    globals()[_name] = _df

print(f"Loaded {len(dataframes)} datasets.")
print("Example variables:", ", ".join(list(dataframes.keys())[:15]))

## 5. Discover the complete nflverse public release catalog

This uses GitHub's public API to inventory release names and asset download URLs. It does not automatically download every file: releases can include multiple formats, large files, archives, and data outside your analysis needs.

In [ ]:
GITHUB_RELEASES_API = "https://api.github.com/repos/nflverse/nflverse-data/releases?per_page=100"

def get_all_github_releases():
    rows = []
    url = GITHUB_RELEASES_API
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "nfl-public-data-notebook"}
    while url:
        response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        page = response.json()
        for release in page:
            for asset in release.get("assets", []):
                rows.append({
                    "release_tag": release.get("tag_name"),
                    "release_name": release.get("name"),
                    "published_at": release.get("published_at"),
                    "asset_name": asset.get("name"),
                    "asset_size_bytes": asset.get("size"),
                    "asset_updated_at": asset.get("updated_at"),
                    "download_count": asset.get("download_count"),
                    "download_url": asset.get("browser_download_url"),
                    "content_type": asset.get("content_type"),
                })
        # GitHub pagination: follow the Link header's rel=next URL.
        link = response.links.get("next", {})
        url = link.get("url")
    return pd.DataFrame(rows)

try:
    nflverse_asset_catalog = get_all_github_releases()
    log_result("nflverse_asset_catalog", "success", f"{len(nflverse_asset_catalog)} assets")
    display(nflverse_asset_catalog.sort_values(["release_tag", "asset_name"], na_position="last").head(100))
except Exception as e:
    nflverse_asset_catalog = pd.DataFrame()
    log_result("nflverse_asset_catalog", "failed", repr(e))
    print("Catalog request failed:", e)

## 6. Optional: save the full release inventory as a CSV

In [ ]:
if not nflverse_asset_catalog.empty:
    nflverse_asset_catalog.to_csv(OUTPUT_DIR / "nflverse_asset_catalog.csv", index=False)
    print("Saved:", OUTPUT_DIR / "nflverse_asset_catalog.csv")

## 7. Optional ESPN scoreboard JSON example

This is an **undocumented endpoint**, provided as a best-effort example. It can change, and access to a URL does not grant reuse rights. The code stores raw JSON and extracts event-level rows if the expected schema is present.

In [ ]:
ESPN_SCOREBOARD_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"

def fetch_espn_scoreboard(params=None):
    r = requests.get(
        ESPN_SCOREBOARD_URL, params=params or {},
        headers={"User-Agent": "Mozilla/5.0 (compatible; NFLDataResearch/1.0)"},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    return r.json()

try:
    espn_scoreboard_json = fetch_espn_scoreboard()
    events = espn_scoreboard_json.get("events", [])
    espn_scoreboard = pd.json_normalize(events) if events else pd.DataFrame()
    log_result("espn_scoreboard", "success", f"{len(espn_scoreboard)} events")
    print("Scoreboard events:", len(espn_scoreboard))
    display(espn_scoreboard.head())
except Exception as e:
    espn_scoreboard_json = None
    espn_scoreboard = pd.DataFrame()
    log_result("espn_scoreboard", "failed", repr(e))
    print("ESPN endpoint unavailable or changed:", e)

## 8. Inspect every DataFrame

Quickly review shape, columns, and sample rows before modeling. Use the `dataframes` dictionary to avoid accidentally mixing grain levels (play, game, player-week, player-season, etc.).

In [ ]:
dataframe_inventory = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 2),
        "column_names": ", ".join(map(str, df.columns[:30])) + (" ..." if len(df.columns) > 30 else "")
    }
    for name, df in dataframes.items()
]).sort_values("rows", ascending=False) if dataframes else pd.DataFrame()

display(dataframe_inventory)

In [ ]:
# Preview a chosen dataset, for example:
if "pbp" in dataframes:
    display(dataframes["pbp"].head())
if "schedules" in dataframes:
    display(dataframes["schedules"].head())

## 9. Save collected DataFrames

Parquet preserves types and is generally preferable for large data. CSV is optional and can consume much more disk space. This writes each dataset separately and records the export manifest.

In [ ]:
export_manifest = []
for name, df in dataframes.items():
    if SAVE_PARQUET:
        path = OUTPUT_DIR / f"{name}.parquet"
        try:
            df.to_parquet(path, index=False)
            export_manifest.append({"dataset": name, "format": "parquet", "path": str(path), "status": "success", "rows": len(df)})
        except Exception as e:
            export_manifest.append({"dataset": name, "format": "parquet", "path": str(path), "status": "failed", "rows": len(df), "detail": repr(e)})
    if SAVE_CSV:
        path = OUTPUT_DIR / f"{name}.csv"
        try:
            df.to_csv(path, index=False)
            export_manifest.append({"dataset": name, "format": "csv", "path": str(path), "status": "success", "rows": len(df)})
        except Exception as e:
            export_manifest.append({"dataset": name, "format": "csv", "path": str(path), "status": "failed", "rows": len(df), "detail": repr(e)})

export_manifest = pd.DataFrame(export_manifest)
display(export_manifest)
export_manifest.to_csv(OUTPUT_DIR / "export_manifest.csv", index=False)

## 10. Save collection log and status

In [ ]:
collection_status.to_csv(OUTPUT_DIR / "collection_status.csv", index=False)
pd.DataFrame(run_log).to_csv(OUTPUT_DIR / "run_log.csv", index=False)
print("Output directory:", OUTPUT_DIR.resolve())
print("Files:", len(list(OUTPUT_DIR.glob("*"))))

## 11. Suggested next integrations

Add new sources as independent functions that return DataFrames and append status rows, rather than merging everything immediately.

- **Official NFL stats and NFL.com:** use published download options and authorized APIs; avoid assuming hidden site endpoints are stable or permitted.
- **ESPN:** schedules, scores, box scores, game logs, QBR and play-by-play via public but undocumented JSON endpoints; schema and terms can change.
- **Pro Football Reference:** historical box scores, player game logs, standings, draft and combine context. Prefer licensed access or existing open mirrors; rate-limit any permitted retrieval.
- **Big Data Bowl / Kaggle:** tracking-data competition files and specialized datasets; some require a Kaggle account and acceptance of competition terms.
- **CollegeFootballData:** college player statistics and recruiting/prospect context; requires an API key.
- **Commercial providers:** Sportradar and SportsDataIO can offer broader feeds under paid agreements.
- **Other GitHub repositories:** discoverable through GitHub search, but verify each dataset's source, date, and license.

### Data engineering notes
1. Preserve source, retrieval timestamp, season, and license metadata.
2. Do not merge on player name alone. Prefer stable IDs (`gsis_id`, `pfr_id`, etc.) and document ID mapping.
3. Understand each table's grain before joining: one row per play is not interchangeable with one row per game or player-season.
4. Validate row counts and latest season/week; a successful download may still be stale or incomplete.
5. Keep raw downloads separate from cleaned/derived tables.
6. Publicly viewable statistics are not necessarily public-domain or licensed for unrestricted redistribution.

## Expanded source inventory: additional public and licensed data

This section expands the original inventory to include sources beyond nflverse. Public access does not necessarily permit bulk scraping or redistribution; check each provider's current terms, API documentation, licenses, and rate limits.

| Data family | Source | Access / caveat |
|---|---|---|
| Official stats, leaders, gamebooks | [NFL.com Stats](https://www.nfl.com/stats/) | Published website data; downloads/APIs vary |
| Tracking-derived metrics | [NFL Next Gen Stats](https://nextgenstats.nfl.com/) | Public summaries; not the same as raw tracking feeds |
| Tracking competition datasets | [NFL Big Data Bowl](https://operations.nfl.com/inside-football-ops/innovation/big-data-bowl/) and [Kaggle](https://www.kaggle.com/) | Competition-specific files; account/rules may apply |
| Historical box scores, game logs, standings, draft | [Pro Football Reference](https://www.pro-football-reference.com/) | Terms, rate limits, and paid features apply |
| Scores, schedules, box scores, QBR, play-by-play | [ESPN NFL](https://www.espn.com/nfl/) | Some public JSON endpoints are undocumented and may change |
| Community datasets and archives | [GitHub NFL data search](https://github.com/search?q=nfl+data&type=repositories) | Verify provenance, freshness, license, and source |
| College stats, recruiting, prospects | [CollegeFootballData](https://collegefootballdata.com/) | API key required; college context rather than NFL game data |
| Weather by venue and kickoff | [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api) | Public API; observe usage terms and limits |
| Betting odds and market history | [The Odds API](https://the-odds-api.com/) | API key; historical data may require paid plan |
| Contract/cap context | [Over The Cap](https://overthecap.com/) and nflverse mirrors | Definitions may be estimates; terms vary |
| Fantasy projections and rankings | [FantasyPros](https://www.fantasypros.com/nfl/) | Website/licensed feeds; availability varies |
| Commercial data feeds | [Sportradar](https://sportradar.com/), [SportsDataIO](https://sportsdata.io/) | Licensed and generally paid |
| Official injury/inactive/practice reports | [NFL.com](https://www.nfl.com/) and team sites | Historical archives may be incomplete |
| College historical player stats | [Sports Reference CFB](https://www.sports-reference.com/cfb/) | Terms/rate limits apply |

**Important gap:** there is no complete, unrestricted public download of every raw NFL tracking coordinate, every All-22 video angle, all proprietary play charting, every historical betting-line snapshot, or all internal league feeds. For those, use authorized competition releases, official licensing, or paid providers rather than bypassing access controls.

## Extra integration A: discover public GitHub NFL repositories

The GitHub search API is rate-limited. This cell provides a discovery list for possible additional datasets; inspect each repository and license before using it.

In [ ]:
def github_repo_search(query="NFL data", per_page=30):
    payload = requests.get(
        "https://api.github.com/search/repositories",
        params={"q": query, "per_page": per_page, "sort": "updated"},
        headers={"Accept": "application/vnd.github+json", "User-Agent": "NFLDataResearchNotebook/1.0"},
        timeout=REQUEST_TIMEOUT,
    )
    payload.raise_for_status()
    items = payload.json().get("items", [])
    return pd.DataFrame([{
        "repository": x.get("full_name"),
        "description": x.get("description"),
        "url": x.get("html_url"),
        "updated_at": x.get("updated_at"),
        "stars": x.get("stargazers_count"),
        "license": (x.get("license") or {}).get("spdx_id")
    } for x in items])

try:
    github_nfl_repositories = github_repo_search("NFL data", 30)
    display(github_nfl_repositories)
    github_nfl_repositories.to_csv(OUTPUT_DIR / "github_nfl_repositories.csv", index=False)
    log_result("github_nfl_repositories", "success", f"{len(github_nfl_repositories)} repositories")
except Exception as e:
    github_nfl_repositories = pd.DataFrame()
    log_result("github_nfl_repositories", "failed", repr(e))
    print("GitHub search unavailable:", e)

## Extra integration B: CollegeFootballData API (optional)

Use this for college player/team context, recruiting and draft-prospect research. It is not a source of NFL game data. Obtain a key from [CollegeFootballData](https://collegefootballdata.com/) and set it as an environment variable named `CFBD_API_KEY`; never paste secret keys into a notebook you share.

In [ ]:
CFBD_API_KEY = os.getenv("CFBD_API_KEY")

def cfbd_get(endpoint, params=None):
    if not CFBD_API_KEY:
        raise RuntimeError("Set CFBD_API_KEY in your environment to enable this source.")
    r = requests.get(
        "https://api.collegefootballdata.com/" + endpoint.lstrip("/"),
        params=params,
        headers={"Authorization": f"Bearer {CFBD_API_KEY}"},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    return r.json()

cfbd_dataframes = {}
if CFBD_API_KEY:
    # Example endpoints; season parameters should be changed to the years you need.
    cfbd_endpoints = {
        "cfbd_teams": ("teams", {}),
        "cfbd_rosters": ("roster", {"year": 2025}),
        "cfbd_draft": ("draft", {"year": 2025}),
        "cfbd_recruiting": ("recruiting/players", {"year": 2025}),
    }
    for name, (endpoint, params) in cfbd_endpoints.items():
        try:
            payload = cfbd_get(endpoint, params)
            df = pd.json_normalize(payload)
            cfbd_dataframes[name] = df
            dataframes[name] = df
            log_result(name, "success", f"{len(df)} rows")
        except Exception as e:
            log_result(name, "failed", repr(e))
else:
    print("CollegeFootballData skipped; CFBD_API_KEY is not set.")

## Extra integration C: historical weather (optional)

Weather can add context to game outcomes, kicking, passing and totals. Build a verified stadium dimension table with latitude/longitude, then retrieve hourly weather near each kickoff. The example below is a helper and does not automatically loop over all games.

In [ ]:
def get_open_meteo_historical_weather(latitude, longitude, start_date, end_date,
                                      timezone_name="America/New_York"):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": timezone_name,
    }
    r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    payload = r.json()
    return pd.DataFrame(payload.get("hourly", {}))

# Example only — coordinates and date range are illustrative:
# weather_example = get_open_meteo_historical_weather(40.8136, -74.0745, "2024-09-01", "2024-09-02")
# display(weather_example.head())

## Extra integration D: betting odds (optional / often paid)

A complete historical odds archive is not generally available as a single free, unrestricted download. Providers such as The Odds API require a key and may charge for historical endpoints. The template below only runs when you explicitly set `ODDS_API_KEY`. Review plan limits and terms before making requests.

In [ ]:
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

def get_current_odds(sport="americanfootball_nfl", regions="us", markets="h2h,spreads,totals"):
    if not ODDS_API_KEY:
        raise RuntimeError("Set ODDS_API_KEY in your environment and review pricing/terms first.")
    r = requests.get(
        f"https://api.the-odds-api.com/v4/sports/{sport}/odds/",
        params={
            "apiKey": ODDS_API_KEY,
            "regions": regions,
            "markets": markets,
            "oddsFormat": "american",
        },
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    return pd.json_normalize(r.json())

# Uncomment only if you have a valid key and intend to use your quota:
# odds_current = get_current_odds()
# dataframes["odds_current"] = odds_current

## Recommended warehouse architecture

Do not merge every source into one giant table. Keep raw source tables separate, then build standardized dimensions and analysis-ready fact tables. This preserves provenance and prevents accidental many-to-many joins.

In [ ]:
warehouse_recommendation = pd.DataFrame([
    {"layer":"raw", "table":"raw_pbp", "grain":"one row per play", "key":"game_id + play_id", "purpose":"Original play-by-play and derived fields"},
    {"layer":"raw", "table":"raw_player_stats", "grain":"source-defined player/game or player/week", "key":"player_id + game_id/week + season", "purpose":"Preserve player statistics as published"},
    {"layer":"dimension", "table":"dim_player", "grain":"one row per canonical player", "key":"canonical_player_id / GSIS ID", "purpose":"Map GSIS, PFR, ESPN and other IDs; never join on name alone"},
    {"layer":"dimension", "table":"dim_team", "grain":"one row per team/franchise version", "key":"team_id + effective dates", "purpose":"Handle team relocations and changing abbreviations"},
    {"layer":"dimension", "table":"dim_game", "grain":"one row per game", "key":"game_id", "purpose":"Kickoff, season/week, venue, home/away, final score"},
    {"layer":"fact", "table":"fact_play", "grain":"one row per play", "key":"game_id + play_id", "purpose":"EPA/WPA, down/distance, field position and play context"},
    {"layer":"fact", "table":"fact_player_game", "grain":"one row per player per game", "key":"player_id + game_id", "purpose":"Box score and advanced statistics"},
    {"layer":"fact", "table":"fact_player_week", "grain":"one row per player per week", "key":"player_id + season + week", "purpose":"Weekly fantasy and performance evaluation"},
    {"layer":"fact", "table":"fact_roster_snapshot", "grain":"player/team/date snapshot", "key":"player_id + team_id + snapshot_date", "purpose":"Injuries, depth charts, roster and availability history"},
    {"layer":"context", "table":"game_weather", "grain":"one row per game", "key":"game_id", "purpose":"Weather at venue around kickoff"},
    {"layer":"context", "table":"game_odds_snapshot", "grain":"game/book/market/timestamp", "key":"game_id + sportsbook + market + timestamp", "purpose":"Opening/closing lines and market movement, if licensed"},
    {"layer":"metadata", "table":"source_manifest", "grain":"one row per dataset retrieval", "key":"source + dataset + retrieved_at", "purpose":"Provenance, license, row count, version and status"},
])
display(warehouse_recommendation)

## Final recommended workflow

1. Run the core nflverse loaders first: play-by-play, schedules, player/team stats, players, rosters, snap counts, participation, injuries, draft/combine, Next Gen Stats, depth charts, contracts, and fantasy datasets where supported.
2. Inspect `collection_status` and `dataframe_inventory`; a failed optional loader should not stop the rest of the collection.
3. Add ESPN and GitHub discovery as supplementary sources; validate any overlap instead of concatenating blindly.
4. Add CollegeFootballData, weather, odds, and licensed providers only when you need that context and have appropriate credentials/permissions.
5. Preserve source tables, retrieval timestamps, licensing notes, and stable identifiers. Avoid player-name-only joins.
6. Validate data grain, duplicate rows, missingness, historical season coverage, and freshness before building models.
7. Use point-in-time snapshots for predictions so later injury reports, closing odds, or post-game information do not leak into pre-game features.

### Known gaps that may require permission or paid access
- Complete raw player tracking coordinates for every play and season.
- Complete All-22 video and proprietary charting / route labels.
- Full historical sportsbook lines and timestamps across books.
- Complete historical injury/practice reports and inactive-list snapshots.
- Internal NFL feeds and some official APIs.
- Some projections, premium statistics, and vendor feeds.

The maintained `nflreadpy` documentation and loader reference are the source of truth for which bulk datasets currently exist:
- https://nflreadpy.nflverse.com/
- https://nflreadpy.nflverse.com/api/load_functions/
- https://github.com/nflverse/nflverse-data/releases
- https://nflreadr.nflverse.com/articles/nflverse_data_schedule.html